## 03-rag

In [78]:
from dotenv import load_dotenv
load_dotenv()

True

In [79]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_LLM_ENDPOINT_PROVIDER")
)

In [80]:
def llm(prompt: str, text_response:bool = True) -> str:
    """Simple function to call the LLM and return the output text.
    
    :param prompt: The input prompt to send to the LLM.
    :type prompt: str
    :param text_response: Whether to return the text response or the full response object.
    :type text_response: bool
    :return: The output text generated by the LLM.
    :rtype: str
    """
    response = openai_client.responses.create(
        model="gemma-4-31b",
        input=[{"role": "user", "content": prompt}]
    )
    if text_response:
        return response.output_text
    return response

We ask a simple question to test the function

In [81]:
llm("What is the capital of France?")

'The capital of France is Paris.'

Now we try to ask to LLM a more-specific question about zoomcamp whithout any context

In [83]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Since I am an AI, I don’t know which specific course you are referring to! However, in almost all cases, the answer is **yes**, but the "how" depends on the type of course.

Here is how to figure out if you can join based on the platform:

### 1. If it is an Online/Self-Paced Course (Udemy, Coursera, Skillshare, etc.)
**Yes!** You can almost always join these at any time. Since the materials are pre-recorded, you can start from the beginning and work at your own pace.

### 2. If it is a Live Cohort/Bootcamp (Zoom, Live Lectures)
*   **Check the Start Date:** If the course has already started, check if they allow "late enrollment."
*   **Catch-up Work:** Many live courses record their sessions. If you join now, you may just need to watch the previous recordings to catch up to the current week.
*   **Contact the Instructor:** Send a quick email or message to the support team asking, *"I just discovered the course; is it still possible to enroll, and how can I catch up on missed material?

The LLM's response is quite generic and not source grounded - now we provide more context like a mini knowledge base

In [84]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

format the prompt with context injection

In [85]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

call the LLM with context == "better" response

In [86]:
answer = llm(prompt)
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.


In [87]:
# pseudo code for RAG show 01-04-dataset.ipynb
def rag(question):
    search_results = search(question) # => returns a list of relevant documents
    context = build_prompt(question, search_results) # => builds a prompt with the question and the search results
    return llm(context) # => calls the LLM with the built prompt and returns the answer

## 04-dataset

In [88]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [89]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [90]:
documents[1100]

{'id': 'f580e98fdc',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': 'Waitress on Windows / Git Bash: "waitress-serve: command not found"',
 'answer': '`pip install waitress` does install `waitress-serve.exe`, but Python\'s `Scripts/` directory may not be on Git Bash\'s `PATH`. The pip output usually warns about this:\n\n```\nWARNING: The script waitress-serve.exe is installed in \'C:\\Users\\<you>\\...\\Scripts\'\nwhich is not on PATH.\n```\n\nTo fix, add that `Scripts` directory to Git Bash\'s `PATH` permanently:\n\n```bash\nnano ~/.bashrc\n# add this line, with the path from the warning:\nexport PATH="/c/Users/<you>/.../Scripts:$PATH"\n```\n\nClose Git Bash and reopen it. `waitress-serve --help` should now work.\n\nIf you\'re using `uv`, this isn\'t an issue because `uv run waitress-serve ...` runs the binary directly from the venv without needing it on `PATH`.'}

we use metadata "course" as filter in search => the idea is to skip the not relevant documents to create a most uselful context inside a prompt

## 05-search

### Create the index

In [91]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

### Test search

In [92]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
# How can I calibrate the boost_dict?
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

Imagine that documents then are pass to LLM as a candidates

### Filter to create a small search space

In [93]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [94]:
[(doc["question"]) for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

### Create a unique function

In [95]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [96]:
search("Where can I find resources to debug my RAG during the course?")

[{'id': 'b7cdde6b25',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Introduction to LLMs and RAG',
  'question': 'Why are we not using Langchain in the course?',
  'answer': "Langchain is a framework for building LLM-powered apps. We're not using it to learn the basics; think of it like learning HTML, CSS, and JavaScript before learning React or Angular."},
 {'id': '86d99bbf21',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Introduction to LLMs and RAG',
  'question': 'Authentication: Why is my OPENAI_API_KEY not found in the Jupyter notebook?',
  'answer': "There are two options to resolve this issue:\n\n**Option 1: Using direnv**\n\n1. Create a `.envrc` file and add your API key.\n2. Run `direnv allow` in the terminal.\n\nIf you encounter the error:\n\n```python\nOpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable\n```\n\n- Install `dotenv` by running:\n\n    ```bash\n    pip inst

## 06-Building a prompt

In [97]:
# always the same!
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [98]:
# this is the template for the prompt that we will send to the LLM
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [99]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [100]:
build_context(search("Where can I find resources to debug my RAG during the course?"))

"Module 1: Introduction to LLMs and RAG\nQ: Why are we not using Langchain in the course?\nA: Langchain is a framework for building LLM-powered apps. We're not using it to learn the basics; think of it like learning HTML, CSS, and JavaScript before learning React or Angular.\n\nModule 1: Introduction to LLMs and RAG\nQ: Authentication: Why is my OPENAI_API_KEY not found in the Jupyter notebook?\nA: There are two options to resolve this issue:\n\n**Option 1: Using direnv**\n\n1. Create a `.envrc` file and add your API key.\n2. Run `direnv allow` in the terminal.\n\nIf you encounter the error:\n\n```python\nOpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable\n```\n\n- Install `dotenv` by running:\n\n    ```bash\n    pip install python-dotenv\n    ```\n\n- Add the following to a cell in the notebook:\n\n    ```python\n    from dotenv import load_dotenv\n    \n    load_dotenv('.envrc')\n    ```\n\n

In [101]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

## 07 - The LLM

In [102]:
response = llm("What is a RAG?", text_response=False)

The response is a Pydantic object.

In [103]:
type(response)

openai.types.responses.response.Response

In [70]:
print("> raw response:", response, "\n")
print("> message:", response.output[0], "\n")
print("> text:", response.output[0].content[0].text[:50], "...", "\n")
print("> text shortcut:", response.output_text[:50], "...", "\n")
print("> usage:", response.usage, "\n")

> raw response: Response(id='resp_xHdrSyxNscDTR1-5Ea7WV8hSjHyzMKM3eGj_qCHUBHtwjMH6m6BGpun6svaMPD9BUPnzUrjFqFjmKCr82UYYPWjar2c5KncXnaYlsWs5cqZg3zzUB9i_OQ74W2fCTs4TYODXknXRyCItKT8mNRVa8k_eMUMWhyesuRWlmyfnZE4sMXBmN9acfnJBwX-kT9nYRHgLBLteqvaoiRoLTF71pkfxi_T44V5kF3Bce1uh267JPRKqOr8FHLn_pkoxh_glLlqjV1CnJQ1H3tuuwQfbSJcDI5vXcPAWh0oIsafbtli56v5X77tSnIerzqcE6-ys_Af0FjNE0UwnIqSxRdn8w-UpPZMt39ykONe4Fs9JS9iZvqGEGmTys5c80zOeFRqJ4WEKhwEO0s1Md7TFguEBVlCUTcUuQhxQmfP2ZddtqCKvNtnovAFFAmCA6rTDeFF_tZrvmWPfWTy0YH6HHnE=', created_at=1781262682.0, error=None, incomplete_details=None, instructions=None, metadata=None, model='gemma-4-31b', object='response', output=[ResponseOutputMessage(id='msg_953a1094bd32c87c', content=[ResponseOutputText(annotations=[], text='**RAG** stands for **Retrieval-Augmented Generation**. \n\nIn simple terms, it is a technique used to give a Large Language Model (LLM)—like GPT-4—access to data that it wasn\'t originally trained on. It allows the AI to look up specific, real-time, or

Compute cost

In [104]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

AttributeError: 'NoneType' object has no attribute 'input_tokens'

In [105]:
# update LLM function

def llm(instructions: str, 
        user_prompt: str, 
        model:str="gemma-4-31b", 
        text_response:bool = True) -> str:
    """Simple function to call the LLM and return the output text.
    
    :param instructions: The instructions for the LLM.
    :type instructions: str
    :param user_prompt: The input prompt from the user.
    :type user_prompt: str
    :param text_response: Whether to return the text response or the full response object.
    :type text_response: bool
    :return: The output text generated by the LLM.
    :rtype: str
    """
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    if text_response:
        return response.output_text
    return response

In [106]:
# Now assemble the RAG

def RAG(question, course="llm-zoomcamp"):
    # find relevant documents
    top_results = search(question, course=course)
    # build the prompt
    prompt = build_prompt(question, top_results)
    # call the LLM
    return llm(INSTRUCTIONS, prompt)

In [108]:
RAG("How do I get a certificate?")


'To get a certificate, you must:\n\n1.  **Finish the course with a "live" cohort**: Certificates are not awarded for the self-paced mode because you must peer-review 3 capstones after submitting your project, which can only be done while the course is running.\n2.  **Pass the Capstone project**: Passing the Capstone is the requirement for receiving the certificate (homework is not mandatory).\n3.  **Provide your official name**: Ensure your official name (as it appears on identification documents) is entered in the second field of your "Edit Course Profile" settings, otherwise the certificate will display a randomly assigned name.'

## 08 - Rag Helper

sources code in `src/`

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()


NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ingest import load_faq_data, build_index
from src.rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_LLM_ENDPOINT_PROVIDER")
)

assistant = RAGBase(
    index=index,
    model="gemma-4-31b",
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.
